In [0]:
%sql
-- Get comprehensive store table statistics
SELECT
  COUNT(*) as total_stores,
  COUNT(DISTINCT store_id) as unique_store_ids,
  COUNT(DISTINCT manager_id) as unique_managers,
  COUNT(DISTINCT state) as unique_states,
  COUNT(DISTINCT store_type) as unique_store_types,
  MIN(opened_year) as earliest_opened,
  MAX(opened_year) as latest_opened,
  SUM(
    CASE
      WHEN store_id IS NULL THEN 1
      ELSE 0
    END
  ) as null_store_id,
  SUM(
    CASE
      WHEN manager_id IS NULL THEN 1
      ELSE 0
    END
  ) as null_manager_id,
  SUM(
    CASE
      WHEN store_name IS NULL THEN 1
      ELSE 0
    END
  ) as null_store_name,
  SUM(
    CASE
      WHEN city IS NULL THEN 1
      ELSE 0
    END
  ) as null_city,
  SUM(
    CASE
      WHEN state IS NULL THEN 1
      ELSE 0
    END
  ) as null_state
FROM
  automobile_repair.bronze.store

# 🏪 Silver Store Transformation

## Bronze Source
`automobile_repair.bronze.store` - 8 columns

## Data Quality Analysis
* **Total Stores**: 50 stores (all unique ✅)
* **Managers**: 15 unique managers (multi-store responsibility)
* **Geographic Spread**: 35 states, 2 store types
* **Opened**: 2015-2024 (9 years)
* **Data Quality**: **PERFECT** ✅
  * 0 NULL values across all columns
  * 0 duplicate store_ids
  * Complete master dimension table

## Silver Transformation
**4 Essential Columns** (down from 8):
* **store_id** - Primary key for joins (order, invoice, estimate, budget, survey)
* **store_name** - Display name in dashboards
* **manager_id** - Manager performance tracking
* **manager_name** - Manager display name

**Removed 4 columns** (not used in any KPI):
* ❌ `city` - No geography-based KPIs
* ❌ `state` - No geography-based KPIs
* ❌ `opened_year` - No tenure/age analysis
* ❌ `store_type` - No store type breakdown in KPIs

**No data cleaning needed** - Just column selection!

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.silver.store AS
SELECT 
  store_id,
  store_name,
  manager_id,
  manager_name
FROM automobile_repair.bronze.store
WHERE store_id IS NOT NULL;

In [0]:
%sql
SELECT * 
FROM automobile_repair.silver.store 
ORDER BY store_id 
LIMIT 50;

In [0]:
%sql
SELECT 
  manager_id,
  manager_name,
  COUNT(*) as stores_managed,
  ARRAY_JOIN(COLLECT_LIST(CAST(store_id AS STRING)), ', ') as store_ids
FROM automobile_repair.silver.store
GROUP BY manager_id, manager_name
ORDER BY stores_managed DESC, manager_name;